In [1]:
import os

REPO_URL = "https://github.com/sinh2206/O_D.git"
REPO_DIR = "O_D"

if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git fetch --all --prune
    !git checkout -B $(git symbolic-ref --short refs/remotes/origin/HEAD | sed 's@^origin/@@')
    !git reset --hard origin/$(git symbolic-ref --short refs/remotes/origin/HEAD | sed 's@^origin/@@')
    %cd ..

%cd {REPO_DIR}
!git log -1 --oneline


Cloning into 'O_D'...
remote: Enumerating objects: 9534, done.
remote: Counting objects: 100% (503/503), done.
remote: Compressing objects: 100% (398/398), done.
remote: Total 9534 (delta 236), reused 364 (delta 103), pack-reused 9031 (from 2)
Receiving objects: 100% (9534/9534), 1009.97 MiB | 51.80 MiB/s, done.
Resolving deltas: 100% (237/237), done.
Updating files: 100% (9023/9023), done.
/kaggle/working/O_D
e0c0454 (HEAD -> main, origin/main, origin/HEAD) 1.9.14


In [2]:
!pwd
!ls


/kaggle/working/O_D
img_error.py  models	       predict.py  README.md	     results   utils
LICENSE       obj-detec.ipynb  public	   requirements.txt  train.py


In [3]:
%pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 87.7 MB/s eta 0:00:00
  Attempting uninstall: cuda-bindings
    Found existing installation: cuda-bindings 13.2.0
    Uninstalling cuda-bindings-13.2.0:
      Successfully uninstalled cuda-bindings-13.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requ

In [4]:
!python train.py \
  --train_data ./public/annotations/train.json \
  --val_data ./public/annotations/val.json \
  --image_dir ./public/train/images \
  --val_image_dir ./public/val/images \
  --checkpoint_dir ./models/

Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth
100%|███████████████████████████████████████| 83.3M/83.3M [00:00<00:00, 181MB/s]
/kaggle/working/O_D/train.py:600: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)
Device: cuda, AMP: True
Train samples: 7500, Val samples: 1500, Classes: ['person', 'car', 'dog', 'cat', 'chair']
Balanced sampling: True
Class weights enabled: True
Class weights: [0.45, 1.0255, 1.2816, 1.508, 0.9407]
Epoch 001/020 | train_loss=1.3771 (cls=0.3569, reg=0.6962, ctr=0.6708) | val_loss=1.1803
Saved best checkpoint: models/best.pth (val_loss=1.1803)
Epoch 002/020 | train_loss=1.0769 (cls=0.2873, reg=0.4881, ctr=0.6564) | val_loss=0.9544
Saved best checkpoint: models/best.pth (val_loss=0.9544)
Epoch 003/020 | train_loss=0.9733 (cls=0.2604, reg=0.421

In [5]:
!python predict.py \
  --image_dir ./public/val/images \
  --output val_predictions.json \
  --model_path ./models/best.pth

Device: cuda
Checkpoint meta: epoch=18, best_val_loss=0.6891636571034472, img_size=640
Predicted images: 1500
Saved JSON: val_predictions.json
Hardcase items: 50
Hardcase summary: results/hardcase_summary.json


In [6]:
!python public/tools/evaluate_predictions.py \
  --ground_truth public/annotations/val.json \
  --predictions val_predictions.json \
  --output val_score.json

{
  "mAP@0.5": 0.722545,
  "performance_points": 20,
  "iou_threshold": 0.5,
  "num_ground_truth_boxes": 2021,
  "num_predictions": 4053,
  "micro_precision": 0.399457,
  "micro_recall": 0.801089,
  "per_class": {
    "person": {
      "ap": 0.782319,
      "num_ground_truth": 1074,
      "num_predictions": 1539,
      "true_positives": 879,
      "false_positives": 660,
      "recall": 0.818436,
      "precision": 0.57115
    },
    "car": {
      "ap": 0.684412,
      "num_ground_truth": 283,
      "num_predictions": 866,
      "true_positives": 220,
      "false_positives": 646,
      "recall": 0.777385,
      "precision": 0.254042
    },
    "dog": {
      "ap": 0.775038,
      "num_ground_truth": 206,
      "num_predictions": 456,
      "true_positives": 176,
      "false_positives": 280,
      "recall": 0.854369,
      "precision": 0.385965
    },
    "cat": {
      "ap": 0.85727,
      "num_ground_truth": 176,
      "num_predictions": 261,
      "true_positives": 155,
      "fal

In [7]:
!python img_error.py

Rendered hardcase images: 50
Source summary: results/hardcase_summary.json
Predictions: val_predictions.json
Output dir: results


In [8]:
# Zip project excluding public
import os
from pathlib import Path

src = Path('/kaggle/working/O_D')
out_zip = Path('/kaggle/working/train.zip')

if not src.exists():
    raise FileNotFoundError(f'Not found: {src}')

cmd = f"cd {src} && zip -r {out_zip} . -x 'public/*' 'public/**'"
print(cmd)
ret = os.system(cmd)
if ret != 0:
    raise RuntimeError(f'zip failed with code {ret}')
print(f'Created: {out_zip}')


cd /kaggle/working/O_D && zip -r /kaggle/working/train.zip . -x 'public/*' 'public/**'
  adding: utils/ (stored 0%)
  adding: utils/model.py (deflated 71%)
  adding: utils/runtime.py (deflated 67%)
  adding: utils/forecast.py (deflated 69%)
  adding: utils/loss.py (deflated 74%)
  adding: utils/config.py (deflated 45%)
  adding: utils/__pycache__/ (stored 0%)
  adding: utils/__pycache__/loss.cpython-312.pyc (deflated 55%)
  adding: utils/__pycache__/config.cpython-312.pyc (deflated 29%)
  adding: utils/__pycache__/nms.cpython-312.pyc (deflated 54%)
  adding: utils/__pycache__/model.cpython-312.pyc (deflated 56%)
  adding: utils/__pycache__/__init__.cpython-312.pyc (deflated 18%)
  adding: utils/process.py (deflated 69%)
  adding: utils/__init__.py (deflated 51%)
  adding: utils/nms.py (deflated 73%)
  adding: utils/image_ops.py (deflated 53%)
  adding: .git/ (stored 0%)
  adding: .git/refs/ (stored 0%)
  adding: .git/refs/tags/ (stored 0%)
  adding: .git/refs/remotes/ (stored 0%)
  add